In [1]:
import sys
sys.path.append('..')
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from src.utils import set_seed, get_device, extract_features, train_clustering

set_seed(42)
device = get_device()
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

In [2]:
print("--- THỰC NGHIỆM 3: TẬP DỮ LIỆU FASHION-MNIST ---")
# Transform ép ảnh 1 kênh thành 3 kênh
transform_fashion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), 
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

fashion_data = torchvision.datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform_fashion)
fashion_subset = torch.utils.data.Subset(fashion_data, range(5000))
fashion_loader = DataLoader(fashion_subset, batch_size=128, shuffle=False)

z_fashion, y_fashion = extract_features(fashion_loader, backbone, device)

--- THỰC NGHIỆM 3: TẬP DỮ LIỆU FASHION-MNIST ---


100%|██████████| 26.4M/26.4M [00:22<00:00, 1.16MB/s]  
100%|██████████| 29.5k/29.5k [00:00<00:00, 177kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.24MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.15MB/s]


In [3]:
k_star_f, acc_f, nmi_f, ari_f = train_clustering(z_fashion, y_fashion, k_max=30, device=device, apply_sparsity=True)
print(f"K_max khởi tạo: 30 | K* tự động tìm được: {k_star_f}")
print(f"ACC = {acc_f*100:.2f}% | NMI = {nmi_f*100:.2f}% | ARI = {ari_f*100:.2f}%")

K_max khởi tạo: 30 | K* tự động tìm được: 2
ACC = 20.28% | NMI = 33.71% | ARI = 15.66%
